# Skyntri — Model A: Facial Skin Concerns (EfficientNet)

This notebook trains the model behind **Skin Analysis → "Facial skin concerns"**. It is written for **Google Colab** (GPU: *Runtime → Change runtime type → T4 GPU*). **It has not been run.** Upload the datasets, then run every cell from top to bottom.

**What it does:** loads the datasets → cleans them (broken, tiny, dark or blurry images, exact and near duplicates, Roboflow copies, conflicting labels) → crops to the face → makes a leakage-free split → applies colour-safe augmentation → trains EfficientNet → tunes decision thresholds on validation → evaluates on the test set with many metrics → exports the files the Skyntri backend loads → runs a prediction exactly like the backend.

## Why these datasets (research summary)

| Dataset | Images | Labels | Verdict |
|---|---|---|---|
| **Skin Issues Dataset v2 (balanced)**, Kaggle | ~9,770 | acne, blackheads, dark spots, pores, wrinkles (one folder per class) | **Primary.** Largest balanced set of facial concern images |
| **Facial Skin (Acne, Pigmentation, Pores, Wrinkles)**, Kaggle | ~4.2k, 8 folders | Redness, dark spots, inflammatory acne, non inflammatory acne black heads / white heads, pigmentation, pores, wrinkles | **Second source.** Adds *redness* and more acne and pigmentation |
| Facial Skin Analysis and Type Classification (killa92), Kaggle | 4,093 | severity 0–5 for acne, pores and others | **Not used.** Severity labels exist for only ~150 rows; the rest only have a skin type |
| Roboflow Skin-Problem-MultiLabel (the old model) | ~4.8k | 10 labels | **Dropped.** Noisy labels and many augmented duplicates |
| ISIC Archive | 400k+ | skin lesions (dermoscopy) | Not facial concerns; used in the **disease** notebook instead |

Download links:
* Skin Issues v2: https://www.kaggle.com/datasets/ahmedismaiil/skin-issues-version-2-dataset-balanced
* Facial Skin (Acne, Pigmentation, Pores, Wrinkles): https://www.kaggle.com/datasets/shijo96john/facial-skin-acne-pigmentation-pores-wrinkles
* Optional, for a *Normal Skin* class: put photos of clear faces that you have checked yourself in a folder named `normal`. For example, use photos from the `normal` folder of https://www.kaggle.com/datasets/shakyadissanayake/oily-dry-and-normal-skin-types-dataset that have no visible acne or spots.

Check each licence on its Kaggle page. These datasets are for education and research.

**Design for the best accuracy:**
* The datasets give **one label per image**, so the model is a **softmax classifier**. It learns which concern is most visible.
* The backend still reports a second concern when its probability passes a threshold tuned on validation data.
* **Face crop:** the model is trained on the face region. The backend crops the same way, so selfies with background work well.
* **EfficientNetB3 at 300 px** by default (B0, B3 or V2-S can be selected), with mixed precision on the GPU.
* **Test-time augmentation:** the original and a flipped copy are averaged. The backend does the same.
* **Abstention threshold:** if the model is not confident, the app asks for a better photo instead of guessing.

## 0. Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Colab already has TensorFlow and every other library these notebooks use, so nothing is installed here.
# Installing another TensorFlow version only causes dependency conflicts. The exported .keras files load
# in the backend, which uses its own TensorFlow (backend/requirements.txt).
if not IN_COLAB:
    print("Outside Colab, install first: pip install tensorflow opencv-python-headless scikit-learn pandas matplotlib seaborn tqdm kaggle")

In [ ]:
import hashlib
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================== CONFIGURATION ==============================
SEED = 42
DATA_ROOT = Path("/content/drive/MyDrive/skyntri/datasets/facial")   # datasets you upload/download
WORK_DIR = Path("/content/drive/MyDrive/skyntri/work/facial")        # cleaned images, checkpoints, logs
EXPORT_DIR = Path("/content/drive/MyDrive/skyntri/export")           # files to copy into backend/models/

# Each sub-folder of DATA_ROOT is one source; class = matching folder name (see CLASS_RULES)
SOURCES = {
    "skin_issues": DATA_ROOT / "skin_issues",            # Kaggle ahmedismaiil/skin-issues-version-2-dataset-balanced
    "facial_skin_8class": DATA_ROOT / "facial_skin_8class",  # Kaggle shijo96john/facial-skin-acne-pigmentation-pores-wrinkles
    "normal": DATA_ROOT / "normal",                      # optional: your checked clear-skin faces
}

# Model ("EfficientNetB0" 224px fast | "EfficientNetB3" 300px best balance | "EfficientNetV2S" 384px slowest)
BACKBONE = "EfficientNetB3"
IMG_SIZE = {"EfficientNetB0": 224, "EfficientNetB3": 300, "EfficientNetV2S": 384}[BACKBONE]
PRETRAINED_WEIGHTS = "imagenet"
BATCH_SIZE = 32
HEAD_EPOCHS = 6
FINE_TUNE_EPOCHS = 30
HEAD_LR = 1e-3
FINE_TUNE_LR = 4e-5
UNFREEZE_FRACTION = 0.6          # fine-tune the top 60% of the backbone
LABEL_SMOOTHING = 0.1
MIXED_PRECISION = True

MODEL_VERSION = f"facial-concerns-{BACKBONE.lower()}-v2"
DATASET_VERSION = time.strftime("%Y-%m-%d")

# Cleaning / preprocessing
MIN_SIDE = 96
NEAR_DUP_HAMMING = 4
MIN_SHARPNESS = 6.0
DARK_LUMA, BRIGHT_LUMA = 30, 240
FACE_CROP = True                 # the backend repeats exactly this crop
FACE_MARGIN = 0.35
CACHE_SIDE = 448
MIN_IMAGES_PER_CLASS = 120       # classes with fewer images are dropped (too little data)
MAX_IMAGES_PER_CLASS = 3000      # cap dominant classes

# Decision rules (tuned on validation)
MIN_PRECISION_FOR_THRESHOLD = 0.55
TARGET_SELECTIVE_ACCURACY = 0.85  # abstain below the confidence that reaches this accuracy
MIN_COVERAGE = 0.80

random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
for p in (WORK_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Folder name keyword -> class. First match wins. Class names match the backend's recommendation table.
CLASS_RULES = [
    ("Blackheads", ["blackhead", "whitehead", "black head", "white head", "comedone", "non inflammatory"]),
    ("Acne", ["acne", "pimple", "papule", "pustule", "inflammatory"]),
    ("Dark Spots", ["dark spot", "dark_spot", "darkspot", "pigment", "melasma", "hyperpig"]),
    ("Pores", ["pore"]),
    ("Wrinkles", ["wrinkle", "fine line", "fine_line"]),
    ("Skin Redness", ["redness", "rosacea", "erythema"]),
    ("Eye bags", ["eye bag", "eyebag", "eye_bag", "dark circle", "dark_circle", "puffy"]),
    ("Normal Skin", ["normal", "clear", "healthy"]),
]
DISPLAY_NAMES = {"Blackheads": "Blackheads & Whiteheads", "Pores": "Visible Pores",
                 "Wrinkles": "Fine Lines & Wrinkles", "Eye bags": "Eye Bags / Dark Circles"}
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

if MIXED_PRECISION and tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

## 1. Get the data

**Option A (Kaggle API, fastest):** in Kaggle open *Settings → API → Create New Token*, upload the downloaded `kaggle.json` when the next cell asks, and let it download and unzip both datasets into `DATA_ROOT`.

**Option B (manual):** download the zip files from the links above, unzip them, and copy them to Google Drive under `MyDrive/skyntri/datasets/facial/skin_issues/` and `.../facial_skin_8class/`.

In [ ]:
USE_KAGGLE_API = True

if USE_KAGGLE_API and IN_COLAB and not any(p.exists() and any(p.iterdir()) for p in [SOURCES["skin_issues"]]):
    from google.colab import files
    print("Upload kaggle.json")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d ahmedismaiil/skin-issues-version-2-dataset-balanced -p "{SOURCES['skin_issues']}" --unzip
    !kaggle datasets download -d shijo96john/facial-skin-acne-pigmentation-pores-wrinkles -p "{SOURCES['facial_skin_8class']}" --unzip

for name, root in SOURCES.items():
    n = sum(1 for p in root.rglob("*") if p.suffix.lower() in IMG_EXT) if root.exists() else 0
    print(f"{name:20s} {n:6d} images   {root}")

### Optional: Normal Skin class

Without this class the model always names some concern, even for clear skin. The next cell downloads the Kaggle *Oily, Dry and Normal Skin Types* dataset (https://www.kaggle.com/datasets/shakyadissanayake/oily-dry-and-normal-skin-types-dataset), copies up to 800 images from its `normal` folders to `DATA_ROOT/normal/` and shows 24 of them. Delete photos with visible acne or spots from that Drive folder before training (keep at least `MIN_IMAGES_PER_CLASS`). Set `ADD_NORMAL_FROM_KAGGLE = False` to skip.

In [ ]:
ADD_NORMAL_FROM_KAGGLE = True
NORMAL_MAX_IMAGES = 800

if ADD_NORMAL_FROM_KAGGLE and IN_COLAB and not (SOURCES["normal"].exists() and any(SOURCES["normal"].iterdir())):
    import os
    import shutil

    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        from google.colab import files
        print("Upload kaggle.json")
        files.upload()
        !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    tmp = Path("/content/oily_dry_normal")
    !kaggle datasets download -d shakyadissanayake/oily-dry-and-normal-skin-types-dataset -p "{tmp}" --unzip -q
    normal_imgs = [p for p in tmp.rglob("*")
                   if p.suffix.lower() in IMG_EXT and any(part.lower() == "normal" for part in p.parts)]
    print("normal images found:", len(normal_imgs))
    random.Random(SEED).shuffle(normal_imgs)
    SOURCES["normal"].mkdir(parents=True, exist_ok=True)
    for i, p in enumerate(normal_imgs[:NORMAL_MAX_IMAGES]):
        shutil.copy(p, SOURCES["normal"] / f"normal_{i:04d}{p.suffix.lower()}")

if SOURCES["normal"].exists():
    shown = sorted(SOURCES["normal"].iterdir())[:24]
    if shown:
        fig, axes = plt.subplots(4, 6, figsize=(15, 10))
        for ax in axes.flat:
            ax.axis("off")
        for ax, p in zip(axes.flat, shown):
            ax.imshow(Image.open(p).convert("RGB"))
        plt.suptitle(f"{len(list(SOURCES['normal'].iterdir()))} images in {SOURCES['normal']}")
        plt.show()

## 2. Build the manifest

The class comes from the folder names in each image's path, for example `.../acne/img1.jpg` or `.../train/dark spots/x.jpg`. Folders that match no rule are listed and skipped. Any train/valid/test split that comes with a download is ignored: we re-split after de-duplication.

In [ ]:
def _norm(text):
    return " ".join(text.lower().replace("-", " ").replace("_", " ").split())


def class_from_path(path):
    parts = [_norm(p) for p in path.parts[:-1]]
    for part in reversed(parts):          # deepest folder first
        for cls, keys in CLASS_RULES:     # first matching rule wins (Blackheads before Acne)
            if any(_norm(k) in part for k in keys):
                return cls
    return None


def roboflow_group_key(name):
    name = Path(name).name
    return (name.split(".rf.")[0] if ".rf." in name else Path(name).stem).lower()


rows, skipped = [], defaultdict(int)
for source, root in SOURCES.items():
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if p.suffix.lower() not in IMG_EXT:
            continue
        # include the source folder name itself, so e.g. DATA_ROOT/normal/*.jpg -> "Normal Skin"
        cls = class_from_path(p.relative_to(root.parent))
        if cls is None:
            skipped[str(p.parent.relative_to(root.parent))] += 1
            continue
        rows.append({"path": str(p), "source": source, "label": cls,
                     "group": f"{source}:{roboflow_group_key(p.name)}"})

manifest = pd.DataFrame(rows)
print("Images with a class:", len(manifest))
if skipped:
    print("Skipped folders (no class rule matched):", dict(skipped))
display(manifest.groupby(["label", "source"]).size().unstack(fill_value=0))

# Check: every downloaded folder and the class it was given
for source, root in SOURCES.items():
    if root.exists():
        folders = manifest[manifest.source == source].assign(folder=lambda m: m.path.map(lambda x: str(Path(x).parent.relative_to(root))))
        print(f"\n{source}:")
        print(folders.groupby(["folder", "label"]).size().to_string())

## 3. Cleaning

In [ ]:
def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def dhash_hex(gray, size=8):
    small = cv2.resize(gray, (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits((small[:, 1:] > small[:, :-1]).flatten()).tobytes().hex()


def inspect(path):
    try:
        with Image.open(path) as im:
            rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
    except Exception as exc:
        return {"ok": False, "reason": f"unreadable:{type(exc).__name__}"}
    if min(rgb.shape[:2]) < MIN_SIDE:
        return {"ok": False, "reason": "too_small"}
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(gray, (256, 256), interpolation=cv2.INTER_AREA)
    return {"ok": True, "reason": None, "md5": file_md5(path), "dhash": dhash_hex(gray),
            "luma": float(g.mean()), "sharpness": float(cv2.Laplacian(g, cv2.CV_64F).var())}


info = pd.DataFrame([inspect(p) for p in tqdm(manifest["path"], desc="inspecting")])
manifest = pd.concat([manifest.reset_index(drop=True), info], axis=1)
print(manifest["reason"].value_counts())
manifest = manifest[manifest["ok"] == True].copy()  # noqa: E712

bad = (manifest.luma < DARK_LUMA) | (manifest.luma > BRIGHT_LUMA) | (manifest.sharpness < MIN_SHARPNESS)
print("Removed for quality (dark / overexposed / blurry):", int(bad.sum()))
manifest = manifest[~bad]

# Exact duplicates: drop copies whose labels disagree, keep one copy otherwise
conflict = manifest.groupby("md5")["label"].nunique()
conflict = set(conflict[conflict > 1].index)
print("Exact duplicates with conflicting labels removed:", len(conflict))
before = len(manifest)
manifest = manifest[~manifest.md5.isin(conflict)].drop_duplicates("md5").reset_index(drop=True)
print("Exact duplicate copies removed:", before - len(manifest))

In [ ]:
# Near duplicates (flipped / brightened / re-encoded copies) -> same group, so they never cross splits
parent = {g: g for g in manifest["group"]}


def find(g):
    while parent[g] != g:
        parent[g] = parent[parent[g]]
        g = parent[g]
    return g


hashes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in manifest["dhash"]])
bits = np.unpackbits(hashes, axis=1)
groups = manifest["group"].tolist()
for band in range(4):
    buckets = defaultdict(list)
    for i, row in enumerate(hashes):
        buckets[bytes(row[band * 2: band * 2 + 2])].append(i)
    for idx in buckets.values():
        sub = bits[idx]
        for a in range(len(idx) - 1):
            dist = (sub[a + 1:] != sub[a]).sum(1)
            for off in np.nonzero(dist <= NEAR_DUP_HAMMING)[0]:
                ra, rb = find(groups[idx[a]]), find(groups[idx[a + 1 + off]])
                if ra != rb:
                    parent[rb] = ra
manifest["group"] = [find(g) for g in groups]

mixed = manifest.groupby("group")["label"].nunique()
mixed = set(mixed[mixed > 1].index)
print("Near-duplicate groups with contradicting labels removed:", len(mixed))
manifest = manifest[~manifest.group.isin(mixed)].reset_index(drop=True)

# Keep one image per near-duplicate group (copies add no information, only bias)
manifest = manifest.sort_values("sharpness", ascending=False).drop_duplicates("group").reset_index(drop=True)
print("Images after de-duplication:", len(manifest))

## 4. Face crop, cache and class balance

In [ ]:
CACHE_DIR = WORK_DIR / f"cleaned_{CACHE_SIDE}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
face_detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


def crop_face(img):
    """Identical to backend app/services/model_io.py: largest frontal face + margin, else unchanged."""
    gray = img.convert("L")
    scale = 1.0
    if max(gray.size) > 800:
        scale = 800.0 / max(gray.size)
        gray = gray.resize((int(gray.width * scale), int(gray.height * scale)), Image.Resampling.BILINEAR)
    arr = np.asarray(gray, dtype=np.uint8)
    min_side = max(40, int(min(arr.shape) * 0.12))
    faces = face_detector.detectMultiScale(cv2.equalizeHist(arr), 1.1, 5, minSize=(min_side, min_side))
    if len(faces) == 0:
        return img, False
    x, y, w, h = [int(v / scale) for v in max(faces, key=lambda f: f[2] * f[3])]
    m = int(FACE_MARGIN * max(w, h))
    return img.crop((max(0, x - m), max(0, y - m), min(img.width, x + w + m), min(img.height, y + h + m))), True


def cache_image(row):
    out = CACHE_DIR / f"{row['md5']}.jpg"
    face = False
    if not out.exists():
        with Image.open(row["path"]) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
        if FACE_CROP:
            img, face = crop_face(img)
        img.thumbnail((CACHE_SIDE, CACHE_SIDE), Image.Resampling.LANCZOS)
        img.save(out, quality=95)
    return str(out), face


res = [cache_image(r) for r in tqdm(manifest.to_dict("records"), desc="crop+cache")]
manifest["clean_path"] = [r[0] for r in res]
manifest["face_found"] = [r[1] for r in res]
print("Face found in", round(100 * manifest.face_found.mean(), 1), "% of images (others are close-ups, kept as-is)")

counts = manifest["label"].value_counts()
dropped = counts[counts < MIN_IMAGES_PER_CLASS]
if len(dropped):
    print("Dropping classes with too few images:", dropped.to_dict())
manifest = manifest[~manifest.label.isin(dropped.index)]
manifest = pd.concat(
    [d.sample(min(len(d), MAX_IMAGES_PER_CLASS), random_state=SEED) for _, d in manifest.groupby("label")]
).reset_index(drop=True)

CLASSES = [c for c, _ in CLASS_RULES if c in set(manifest.label)]
NUM_CLASSES = len(CLASSES)
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}
manifest["y"] = manifest["label"].map(CLASS_INDEX)
manifest.to_pickle(WORK_DIR / "manifest_clean.pkl")
print("Classes:", CLASSES)
display(manifest["label"].value_counts())

In [ ]:
# Visual check: 4 random images per class (after face crop)
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(12, 3 * NUM_CLASSES))
for r, cls in enumerate(CLASSES):
    samples = manifest[manifest.label == cls].sample(min(4, (manifest.label == cls).sum()), random_state=SEED)
    for c in range(4):
        ax = axes[r, c] if NUM_CLASSES > 1 else axes[c]
        ax.axis("off")
        if c < len(samples):
            ax.imshow(Image.open(samples.iloc[c]["clean_path"]))
            ax.set_title(cls, fontsize=9)
plt.tight_layout(); plt.show()

## 5. Leakage-free split (grouped + stratified, about 70/15/15)

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = list(sgkf.split(manifest, manifest["y"], manifest["group"]))
manifest["split"] = "train"
manifest.loc[folds[0][1], "split"] = "test"
manifest.loc[folds[1][1], "split"] = "val"
assert manifest.groupby("group")["split"].nunique().max() == 1, "a group crosses splits"
display(manifest.groupby(["label", "split"]).size().unstack(fill_value=0))
manifest.to_csv(WORK_DIR / "manifest_split.csv", index=False)

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
val_df = manifest[manifest.split == "val"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)

## 6. Input pipeline and augmentation

Augmentation is **colour-safe**: there is no hue shift and brightness/contrast change only a little, because colour carries the meaning for *redness* and *dark spots*. Geometric changes (flip, small rotation, zoom, shift) simulate different selfie framing.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05, fill_mode="reflect"),
    tf.keras.layers.RandomZoom((-0.15, 0.10), fill_mode="reflect"),
    tf.keras.layers.RandomTranslation(0.06, 0.06, fill_mode="reflect"),
    tf.keras.layers.RandomContrast(0.12),
    tf.keras.layers.RandomBrightness(0.08, value_range=(0, 255)),
], name="augment")


def load_image(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.cast(tf.image.resize(img, (IMG_SIZE, IMG_SIZE), method="bilinear"), tf.float32)  # 0..255 like backend


def make_ds(df, training):
    ys = tf.one_hot(df["y"].values, NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((df["clean_path"].values, ys))
    if training:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (tf.clip_by_value(augment(x, training=True), 0, 255), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)

x0 = load_image(train_df.loc[0, "clean_path"])[None]
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for i, ax in enumerate(axes):
    img = x0[0] if i == 0 else tf.clip_by_value(augment(x0, training=True)[0], 0, 255)
    ax.imshow(tf.cast(img, tf.uint8)); ax.axis("off"); ax.set_title("original" if i == 0 else f"augmented {i}")
plt.show()

## 7. Model

In [ ]:
class_weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["y"].values)
class_weights = {i: float(min(w, 8.0)) for i, w in enumerate(class_weights)}
print({CLASSES[i]: round(w, 2) for i, w in class_weights.items()})


def build_model(weights=PRETRAINED_WEIGHTS):
    base = getattr(tf.keras.applications, BACKBONE)(
        include_top=False, weights=weights, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32", name="concerns")(x)
    return tf.keras.Model(inputs, outputs, name=f"skyntri_facial_{BACKBONE.lower()}"), base


def predict_probs(m, ds, tta=False):
    out = []
    for x, _ in ds:
        p = m.predict_on_batch(x)
        if tta:
            p = (p + m.predict_on_batch(tf.reverse(x, axis=[2]))) / 2.0
        out.append(p)
    return np.concatenate(out, axis=0)


class MacroF1(tf.keras.callbacks.Callback):
    """Validation macro-F1 each epoch; used to keep the best checkpoint."""

    def __init__(self, ds, y):
        super().__init__()
        self.ds, self.y = ds, y

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        pred = predict_probs(self.model, self.ds).argmax(1)
        logs["val_macro_f1"] = f1_score(self.y, pred, average="macro", zero_division=0)
        print(f" - val_macro_f1: {logs['val_macro_f1']:.4f}")


model, base = build_model()
LOSS = tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)
METRICS = [tf.keras.metrics.CategoricalAccuracy(name="acc"), tf.keras.metrics.TopKCategoricalAccuracy(k=2, name="top2")]
CKPT = WORK_DIR / "best_facial.keras"


def callbacks(patience):
    return [
        MacroF1(val_ds, val_df["y"].values),
        tf.keras.callbacks.ModelCheckpoint(str(CKPT), monitor="val_macro_f1", mode="max", save_best_only=True),
        tf.keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max", patience=patience, restore_best_weights=True),
        tf.keras.callbacks.CSVLogger(str(WORK_DIR / "training_log.csv"), append=True),
    ]


model.summary(show_trainable=True)

## 8. Training (run this yourself)

In [ ]:
# Phase 1: train only the new head (backbone frozen)
model.compile(optimizer=tf.keras.optimizers.AdamW(HEAD_LR, weight_decay=1e-4), loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, class_weight=class_weights, callbacks=callbacks(3))

# Phase 2: fine-tune the top of the backbone (BatchNorm stays frozen)
base.trainable = True
cut = int(len(base.layers) * (1 - UNFREEZE_FRACTION))
for i, layer in enumerate(base.layers):
    layer.trainable = i >= cut and not isinstance(layer, tf.keras.layers.BatchNormalization)

steps = FINE_TUNE_EPOCHS * max(1, math.ceil(len(train_df) / BATCH_SIZE))
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        tf.keras.optimizers.schedules.CosineDecay(FINE_TUNE_LR, steps, alpha=0.05), weight_decay=1e-5),
    loss=LOSS, metrics=METRICS)
model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, class_weight=class_weights, callbacks=callbacks(7))

log = pd.read_csv(WORK_DIR / "training_log.csv")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
log[["loss", "val_loss"]].plot(ax=a1, title="Loss"); log[["acc", "val_acc", "val_macro_f1"]].plot(ax=a2, title="Accuracy / macro-F1")
plt.show()

## 9. Decision rules from the validation set (the test set is never used here)

* **Per-class threshold:** the probability at which the backend also reports a class (for example a *second* concern). It is the F1-optimal value with precision ≥ `MIN_PRECISION_FOR_THRESHOLD`.
* **`abstain_below`:** if the top probability is lower than this, the app says *"we cannot confidently analyze this image"*. It is the lowest confidence that still gives `TARGET_SELECTIVE_ACCURACY` while keeping at least `MIN_COVERAGE` of the images.

In [ ]:
best = tf.keras.models.load_model(CKPT, compile=False)
USE_TTA = True
p_val = predict_probs(best, val_ds, tta=USE_TTA)
y_val = val_df["y"].values

thresholds = {}
for k, cls in enumerate(CLASSES):
    yk = (y_val == k).astype(int)
    if yk.sum() == 0:
        thresholds[cls] = 0.5
        continue
    prec, rec, thr = precision_recall_curve(yk, p_val[:, k])
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-9)
    ok = prec[:-1] >= MIN_PRECISION_FOR_THRESHOLD
    i = int(np.argmax(np.where(ok, f1[:-1], -1))) if ok.any() else int(np.argmax(f1[:-1]))
    thresholds[cls] = float(np.clip(thr[i], 0.15, 0.9))

conf, correct = p_val.max(1), p_val.argmax(1) == y_val
abstain_below = 0.0
for t in np.linspace(0.2, 0.9, 71):
    kept = conf >= t
    if kept.mean() < MIN_COVERAGE:
        break
    if correct[kept].mean() >= TARGET_SELECTIVE_ACCURACY:
        abstain_below = float(t)
        break
print("Thresholds:", {k: round(v, 3) for k, v in thresholds.items()})
print("abstain_below:", round(abstain_below, 3), "coverage:", round(float((conf >= abstain_below).mean()), 3))

cov, acc = [], []
for t in np.linspace(0, 0.95, 40):
    k = conf >= t
    cov.append(k.mean()); acc.append(correct[k].mean() if k.any() else np.nan)
plt.plot(cov, acc, marker="."); plt.xlabel("coverage"); plt.ylabel("accuracy on answered images")
plt.title("Selective prediction (validation)"); plt.grid(alpha=.3); plt.show()

## 10. Test evaluation

In [ ]:
p_plain = predict_probs(best, test_ds, tta=False)
p_test = predict_probs(best, test_ds, tta=USE_TTA)
y_test = test_df["y"].values
y_pred = p_test.argmax(1)

summary = {
    "test_images": int(len(y_test)),
    "accuracy": accuracy_score(y_test, y_pred),
    "accuracy_without_tta": accuracy_score(y_test, p_plain.argmax(1)),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "top2_accuracy": top_k_accuracy_score(y_test, p_test, k=2, labels=np.arange(NUM_CLASSES)),
    "macro_roc_auc_ovr": roc_auc_score(np.eye(NUM_CLASSES)[y_test], p_test, average="macro", multi_class="ovr"),
}
kept = p_test.max(1) >= abstain_below
summary["coverage_after_abstention"] = float(kept.mean())
summary["accuracy_on_answered"] = float((y_pred[kept] == y_test[kept]).mean()) if kept.any() else float("nan")

rows = []
for k, cls in enumerate(CLASSES):
    t, p = y_test == k, y_pred == k
    tp, fn, fp, tn = (t & p).sum(), (t & ~p).sum(), (~t & p).sum(), (~t & ~p).sum()
    rows.append({"class": cls, "support": int(t.sum()),
                 "precision": tp / max(tp + fp, 1), "recall/sensitivity": tp / max(tp + fn, 1),
                 "specificity": tn / max(tn + fp, 1), "f1": f1_score(t, p, zero_division=0),
                 "roc_auc": roc_auc_score(t, p_test[:, k]) if 0 < t.sum() < len(t) else np.nan,
                 "threshold": thresholds[cls]})
per_class = pd.DataFrame(rows).set_index("class")
display(per_class.round(3))
print(classification_report(y_test, y_pred, labels=np.arange(NUM_CLASSES), target_names=CLASSES, zero_division=0))
pd.Series(summary).round(4)

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=np.arange(NUM_CLASSES), normalize="true")
plt.figure(figsize=(1.2 * NUM_CLASSES + 3, 1.0 * NUM_CLASSES + 2))
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=CLASSES, yticklabels=CLASSES, cmap="Blues")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix (row-normalised)")
plt.xticks(rotation=35, ha="right"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 6))
for k, cls in enumerate(CLASSES):
    t = (y_test == k).astype(int)
    if 0 < t.sum() < len(t):
        f, tp_, _ = roc_curve(t, p_test[:, k])
        plt.plot(f, tp_, label=f"{cls} ({roc_auc_score(t, p_test[:, k]):.2f})")
plt.plot([0, 1], [0, 1], "k--"); plt.legend(fontsize=8); plt.title("One-vs-rest ROC (test)"); plt.show()


def ece(conf, correct, bins=10):
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)


summary["ece_top1"] = ece(p_test.max(1), (y_pred == y_test).astype(float))
mids, accs = [], []
for lo in np.linspace(0, 0.9, 10):
    m = (p_test.max(1) > lo) & (p_test.max(1) <= lo + 0.1)
    if m.sum() >= 5:
        mids.append(p_test.max(1)[m].mean()); accs.append((y_pred == y_test)[m].mean())
plt.plot(mids, accs, "o-"); plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("confidence"); plt.ylabel("accuracy")
plt.title(f"Reliability diagram (ECE={summary['ece_top1']:.3f})"); plt.show()

In [ ]:
# Bootstrap 95% confidence intervals (resampling whole groups)
def bootstrap(fn, n=1000):
    rng = np.random.default_rng(SEED)
    gidx = test_df.groupby("group").indices
    keys = list(gidx)
    vals = [fn(np.concatenate([gidx[k] for k in rng.choice(keys, len(keys), replace=True)])) for _ in range(n)]
    return [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))]


ci = {
    "accuracy": bootstrap(lambda i: accuracy_score(y_test[i], y_pred[i])),
    "macro_f1": bootstrap(lambda i: f1_score(y_test[i], y_pred[i], average="macro", zero_division=0)),
    "balanced_accuracy": bootstrap(lambda i: balanced_accuracy_score(y_test[i], y_pred[i])),
}
for k, v in ci.items():
    print(f"{k}: {summary[k]:.3f}  (95% CI {v[0]:.3f} - {v[1]:.3f})")

In [ ]:
# Fairness check: approximate skin tone from the Individual Typology Angle (ITA) of skin pixels.
# ITA is only an approximation - use it to find gaps, not to certify fairness.
def ita_group(path):
    rgb = np.asarray(Image.open(path).convert("RGB").resize((128, 128)))
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    L, b = lab[..., 0] * 100 / 255, lab[..., 2] - 128
    ycrcb = cv2.cvtColor(rgb, cv2.COLOR_RGB2YCrCb)
    mask = (ycrcb[..., 1] >= 133) & (ycrcb[..., 1] <= 180) & (ycrcb[..., 2] >= 75) & (ycrcb[..., 2] <= 130)
    if mask.mean() < 0.05:
        return "unknown"
    ita = np.degrees(np.arctan2(np.median(L[mask]) - 50, np.median(b[mask])))
    return "light (ITA>41)" if ita > 41 else "intermediate (10-41)" if ita > 10 else "dark (ITA<=10)"


test_df["tone"] = [ita_group(p) for p in tqdm(test_df["clean_path"], desc="ITA")]
fair = pd.DataFrame([
    {"group": g, "n": len(i), "accuracy": accuracy_score(y_test[i], y_pred[i]),
     "macro_f1": f1_score(y_test[i], y_pred[i], average="macro", zero_division=0)}
    for g, i in test_df.groupby("tone").indices.items() if len(i) >= 15])
display(fair)
display(pd.DataFrame([{"source": s, "n": len(i), "accuracy": accuracy_score(y_test[i], y_pred[i])}
                      for s, i in test_df.groupby("source").indices.items()]))
display(pd.DataFrame([{"face_found": f, "n": len(i), "accuracy": accuracy_score(y_test[i], y_pred[i])}
                      for f, i in test_df.groupby("face_found").indices.items()]))

In [ ]:
# Error gallery: most confident mistakes (look for label noise and confusing classes)
wrong = np.where(y_pred != y_test)[0]
wrong = wrong[np.argsort(-p_test[wrong].max(1))][:12]
if len(wrong):
    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    for ax, i in zip(axes.flat, wrong):
        ax.imshow(Image.open(test_df.loc[i, "clean_path"])); ax.axis("off")
        ax.set_title(f"true {CLASSES[y_test[i]]}\npred {CLASSES[y_pred[i]]} {p_test[i].max():.2f}", fontsize=8)
    for ax in axes.flat[len(wrong):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

## 11. Export for the Skyntri backend

Copy **both files** from `MyDrive/skyntri/export/` into **`backend/models/`** of the project:

| File | Loaded by |
|---|---|
| `facial_concerns_efficientnet.keras` | `backend/app/services/skin_analysis.py` → `load_dataset3_model()` (setting `DATASET3_MODEL_PATH`) |
| `facial_concerns_metadata.json` | `backend/app/services/skin_analysis.py` → `facial_model_config()` (setting `FACIAL_MODEL_METADATA_PATH`) |

These are the default names, so no setting needs to change. With Docker, run `docker compose restart backend` afterwards.

In [ ]:
MODEL_FILE = EXPORT_DIR / "facial_concerns_efficientnet.keras"
META_FILE = EXPORT_DIR / "facial_concerns_metadata.json"

# Save a float32 copy (mixed precision is only for GPU training; the backend runs on CPU)
tf.keras.mixed_precision.set_global_policy("float32")
export_model, _ = build_model()  # ImageNet build = same layers as in training (weights=None drops a Rescaling layer)
export_model.set_weights(best.get_weights())
export_model.save(MODEL_FILE)
sha256 = hashlib.sha256(MODEL_FILE.read_bytes()).hexdigest()

metadata = {
    "task": "facial_concerns",
    "model_version": MODEL_VERSION,
    "backbone": BACKBONE,
    "image_size": IMG_SIZE,
    "output": "softmax",
    "preprocessing": "RGB, face crop (Haar, margin), bilinear resize to image_size, float32 0-255 (no /255)",
    "face_crop": {"enabled": FACE_CROP, "margin": FACE_MARGIN},
    "tta": USE_TTA,
    "classes": CLASSES,
    "display_names": {c: DISPLAY_NAMES.get(c, c) for c in CLASSES},
    "thresholds": thresholds,
    "abstain_below": abstain_below,
    "sha256": sha256,
    "dataset_version": DATASET_VERSION,
    "sources": sorted(manifest["source"].unique().tolist()),
    "split_sizes": {s: int((manifest.split == s).sum()) for s in ("train", "val", "test")},
    "test_metrics": {k: round(float(v), 4) for k, v in summary.items()},
    "test_metrics_ci95": ci,
    "per_class": per_class.round(4).reset_index().to_dict(orient="records"),
    "notes": "AI estimate of the most visible facial skin-care concern. Not a diagnosis.",
}
META_FILE.write_text(json.dumps(metadata, indent=2, default=float))
per_class.to_csv(EXPORT_DIR / "facial_concerns_per_class_metrics.csv")

card = ["# Dataset card - Skyntri facial concerns", "", f"Version: {DATASET_VERSION}", "",
        "| Source | Images | Licence | Notes |", "|---|---|---|---|"]
for s, n in manifest["source"].value_counts().items():
    card.append(f"| {s} | {n} | check on Kaggle page | |")
card += ["", "Cleaning: unreadable/small/dark/blurry removed, exact + near duplicates removed, contradicting labels removed,",
         "face crop, per-class cap. Split: StratifiedGroupKFold ~70/15/15 (groups never cross splits)."]
(EXPORT_DIR / "facial_concerns_dataset_card.md").write_text("\n".join(card))
print("Saved", MODEL_FILE, META_FILE)

## 12. Prediction, exactly as the backend does it

RGB → face crop → resize → 0..255 → model (+ flipped copy, averaged) → abstain when not confident → top concern plus any other class above its threshold.

In [ ]:
loaded = tf.keras.models.load_model(MODEL_FILE, compile=False)
meta = json.loads(META_FILE.read_text())


def prepare(path):
    with Image.open(path) as im:
        img = ImageOps.exif_transpose(im).convert("RGB")
    if meta["face_crop"]["enabled"]:
        img, _ = crop_face(img)
    arr = np.asarray(img.resize((meta["image_size"],) * 2, Image.Resampling.BILINEAR), dtype=np.float32)
    batch = [arr, arr[:, ::-1, :]] if meta["tta"] else [arr]
    return np.stack(batch)


def predict_concerns(path):
    probs = loaded.predict(prepare(path), verbose=0).mean(axis=0)
    if probs.max() < meta["abstain_below"]:
        return {"status": "uncertain", "message": "We cannot confidently analyze this image. Please upload another photo."}
    found = [c for c, p in zip(meta["classes"], probs) if p >= meta["thresholds"][c]]
    top = meta["classes"][int(probs.argmax())]
    if top not in found:
        found = [top] + found
    concerns = [meta["display_names"].get(c, c) for c in found if c != "Normal Skin"]
    return {
        "status": "completed",
        "concerns": concerns or ["No supported facial concern detected"],
        "probabilities": {c: round(float(p), 3) for c, p in zip(meta["classes"], probs)},
        "model_version": meta["model_version"],
    }


xb, _ = next(iter(test_ds))
# `best` runs in mixed precision (float16), the exported copy in float32: small differences are expected
p_train, p_export = np.asarray(best.predict_on_batch(xb), np.float32), np.asarray(loaded.predict_on_batch(xb), np.float32)
same_top = float((p_train.argmax(1) == p_export.argmax(1)).mean())
print(f"export check: max prob difference {np.abs(p_train - p_export).max():.4f}, same top class {same_top:.1%}")
assert same_top >= 0.95 and np.abs(p_train - p_export).max() < 0.05, "exported model differs from the trained one"
predict_concerns(test_df.loc[0, "path"])   # try any photo: predict_concerns("/content/my_selfie.jpg")